In [1]:
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
import seaborn as sns
import random



In [2]:
file = '2018-09-13--06.02.24.events.csv'

# reads from from the 5th line
sim_data = pd.read_csv(file, header=4)

C:\Users\yysma\AppData\Local\Temp\ipykernel_1700\873465498.py:4: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  sim_data = pd.read_csv(file, header=4)


In [3]:
# checking data type for column 4 due to warning
sim_data["Info"].apply(type).value_counts()

Info
<class 'float'>    61237
<class 'str'>      24249
Name: count, dtype: int64

In [4]:
# converting column 4 to all str type
sim_data["Info"] = sim_data["Info"].astype(str)


In [5]:
# select HCV status for agents at entry (Event = 'activated')
entry = sim_data[sim_data['Event']=='activated']['HCV']
# the proportion of agents in each HCV status at entry
entry_prop = entry.value_counts(normalize=True) # normalize=True converts counts to fractions summing to 1
entry_prop

HCV
susceptible        0.645280
chronic            0.230237
recovered          0.107193
infectiousacute    0.016986
exposed            0.000303
Name: proportion, dtype: float64

In [6]:
exit = sim_data[sim_data['Event']=='deactivated']['HCV']
exit_prop = exit.value_counts(normalize=True)
exit_prop



HCV
susceptible        0.542651
chronic            0.301181
recovered          0.120472
vaccinated         0.032415
infectiousacute    0.003150
exposed            0.000131
Name: proportion, dtype: float64

In [7]:
# excluded vaccinated; not part of disease progression
hcv_mapping = {
    'susceptible': 0,
    'exposed': 1,
    'infectiousacute': 2,
    'chronic': 3,
    'recovered': 4,
}
# adds HCV_num variable by mapping HCV states to their corresponding values
sim_data['HCV_num'] = sim_data['HCV'].map(hcv_mapping)

# select 'activated' rows, set 'Agent' as index, get HCV_num
entry = sim_data[sim_data['Event'] == 'activated'].set_index('Agent')['HCV_num']
# same for 'deactivated' rows 
exit = sim_data[sim_data['Event'] == 'deactivated'].set_index('Agent')['HCV_num']

mean_entry_hcv = entry.mean()
mean_exit_hcv = exit.mean()

print("Mean HCV at entry:", mean_entry_hcv)
print("Mean HCV at exit:", mean_exit_hcv)

Mean HCV at entry: 1.1537607269056032
Mean HCV at exit: 1.438491794384918


In [10]:
# checking whether column is static of time-dependent per agent
static_cols = [
     'DBLabel' # change variable name to check
]

violations = {}

for agent_id, g in sim_data.groupby('Agent'):
    for col in static_cols:
        if g[col].nunique(dropna=True) > 1:
            violations.setdefault(agent_id, []).append(col)

len(violations)

2

In [11]:
sim_data.groupby('Agent')['DBLabel'].nunique(dropna=True).value_counts()


DBLabel
1    39616
2        2
Name: count, dtype: int64

In [12]:
dblabel_weird_agents = (
    sim_data
    .groupby('Agent')['DBLabel']
    .nunique(dropna=True)
    .loc[lambda x: x > 1]
)

dblabel_weird_agents
rows_to_drop = [31330, 4816]
sim_data = sim_data.drop(index=rows_to_drop)

In [13]:
#agent_histories[651156501]
#agent_histories[1164664992]    


In [14]:
# updating agent histories after dropping those two rows
agent_histories = {
    agent_id: group.sort_values('Time')
    for agent_id, group in sim_data.groupby('Agent')
}

In [ ]:
static_vars = [
    'Gender', 
    'Race', 
    'Zip', 
    'Age_Started', 
    'Drug_in_degree',
    'Drug_out_degree',
    'Fraction_recept_sharing',
    'Daily_injection_intensity',
    'DBLabel', 
    'Syringe_source']


In [50]:
activated_agents = (
    sim_data
    .query("Event == 'activated'")
    .drop_duplicates(subset='Agent')
    .copy()
)
deactivated_agents = (
    sim_data
    .query("Event == 'deactivated'")
    .drop_duplicates(subset='Agent')
    .copy()
)

In [51]:
activated_agents['HCV'].value_counts()

HCV
susceptible        25565
chronic             9121
recovered           4247
infectiousacute      673
exposed               12
Name: count, dtype: int64

In [53]:
deactivated_agents['HCV'].value_counts()

HCV
susceptible        4135
chronic            2295
recovered           918
vaccinated          247
infectiousacute      24
exposed               1
Name: count, dtype: int64

In [110]:
filtered_activated = activated_agents[activated_agents['HCV'].isin(['chronic', 'recovered', 'infectiousacute', 'susceptible'])]
filtered_deactivated = deactivated_agents[deactivated_agents['HCV'].isin(['chronic', 'recovered', 'vaccinated', 'susceptible'])]
filtered_activated = filtered_activated.copy()
filtered_deactivated = filtered_deactivated.copy()
filtered_activated['Syringe_source_numeric'] = filtered_activated['Syringe_source'].map({
    'nonHR': 0,
    'HR': 1
})
filtered_deactivated['Syringe_source_numeric'] = filtered_deactivated['Syringe_source'].map({
    'nonHR': 0,
    'HR': 1
})

In [ ]:
mean_activated = (
    filtered_activated
    .groupby('HCV')[['Fraction_recept_sharing', 'Syringe_source_numeric','Num Buddies','Age']]
    .mean()
    .rename(columns={
        'Fraction_recept_sharing': 'Recept Sharing',
        'Syringe_source_numeric': 'Syringe Source'
    }).round(2)
)

mean_deactivated = (
    filtered_deactivated
    .groupby('HCV')[['Fraction_recept_sharing', 'Syringe_source_numeric', 'Num Buddies', 'Age']]
    .mean()
    .rename(columns={
        'Fraction_recept_sharing': 'Recept Sharing',
        'Syringe_source_numeric': 'Syringe Source'
    }).round(2)
)

print(mean_activated)
print(mean_deactivated)

                 Recept_sharing  Syringe_source  Num Buddies    Age
HCV                                                                
chronic                    0.20            0.48         0.66  40.78
infectiousacute            0.21            0.46         0.30  32.19
recovered                  0.19            0.49         0.65  40.30
susceptible                0.20            0.49         0.55  29.94
             Recept_sharing  Syringe_source  Num Buddies    Age
HCV                                                            
chronic                0.23            0.43         0.98  44.80
recovered              0.20            0.49         0.81  44.05
susceptible            0.17            0.51         0.70  35.32
vaccinated             0.23            0.40         2.54  33.62


In [129]:
categorical_static_vars = ['Gender', 'Race', 'Zip']
for col in categorical_static_vars:
    print(f"{col} distribution (%):")
    print(deactivated_agents[col].value_counts(normalize=True) * 100)
    print()

Gender distribution (%):
Gender
Male      68.622047
Female    31.377953
Name: proportion, dtype: float64

Race distribution (%):
Race
NHWhite     58.425197
NHBlack     20.538058
Hispanic    18.254593
Other        2.782152
Name: proportion, dtype: float64

Zip distribution (%):
Zip
60647    4.593176
60651    3.162730
60636    2.703412
60804    2.493438
60639    2.296588
           ...   
60407    0.013123
61270    0.013123
60430    0.013123
60035    0.013123
60478    0.013123
Name: proportion, Length: 314, dtype: float64



In [141]:
corr_list = []

# except for Num Buddies and Age_Started
all_numeric_vars = ['Drug_in_degree', 'Drug_out_degree', 'Fraction_recept_sharing', 
                    'Daily_injection_intensity', 'Age', 'HCV_friend_preval']

for status, group_df in filtered_deactivated.groupby('HCV'):
    corr_matrix = group_df[all_numeric_vars].corr(method='spearman')
    corr_long = (
        corr_matrix
        .stack()                    
        .reset_index()
        .rename(columns={'level_0': 'var1', 'level_1': 'var2', 0: 'spearman_corr'}))
    corr_long['HCV'] = status
    corr_list.append(corr_long)
corr_df = pd.concat(corr_list, ignore_index=True)

In [142]:
# keeps only strong correlations
strong_corrs = corr_df[abs(corr_df['spearman_corr']) > 0.3]
# remove 1.0 correlations 
strong_corrs = strong_corrs[strong_corrs['spearman_corr'] != 1.0]

# remove duplicate variable pairs 
strong_corrs['var_pair'] = strong_corrs.apply(
    lambda row: tuple(sorted([row['var1'], row['var2']])), axis=1
)

strong_corrs_filtered = strong_corrs.drop_duplicates(subset=['var_pair', 'HCV'])
strong_corrs_filtered = strong_corrs_filtered.drop(columns=['var_pair'])
strong_corrs_filtered.reset_index(drop=True, inplace=True)

strong_corrs_filtered.tail()

,var1,var2,spearman_corr,HCV
10,Drug_out_degree,HCV_friend_preval,0.534515,recovered
11,Drug_in_degree,Drug_out_degree,0.331905,susceptible
12,Drug_in_degree,HCV_friend_preval,0.440169,susceptible
13,Drug_out_degree,HCV_friend_preval,0.467755,susceptible
14,Fraction_recept_sharing,Daily_injection_intensity,-0.381055,vaccinated


In [135]:
sim_data = sim_data.sort_values(['Agent', 'Time'])

# get "next state" for each agent
sim_data['next_HCV'] = sim_data.groupby('Agent')['HCV'].shift(-1)

# keep only rows where there is a next state
df_transitions = sim_data.dropna(subset=['next_HCV'])

# calculate proportion for each transition
transition_counts = df_transitions.groupby(['HCV', 'next_HCV']).size()
transition_props = transition_counts / transition_counts.groupby(level=0).sum()

print(transition_props)

HCV              next_HCV       
chronic          chronic            1.000000
exposed          exposed            0.005437
                 infectiousacute    0.994563
infectiousacute  chronic            0.407875
                 infectiousacute    0.059897
                 recovered          0.532229
recovered        exposed            0.002909
                 infectiousacute    0.431839
                 recovered          0.565252
susceptible      chronic            0.002532
                 exposed            0.247435
                 infectiousacute    0.018654
                 recovered          0.001199
                 susceptible        0.550966
                 vaccinated         0.179214
vaccinated       exposed            0.016034
                 infectiousacute    0.000104
                 vaccinated         0.983862
dtype: float64
